# LegalIR Task 1: Colab Single-T4 Contract Smoke Gate
## UIT Data Science Challenge 2026 — Single-GPU Topology Verification
**Pinned Git Commit:** `3792b13699f4706c5a698b2d147a55e97fe4c0ce`

### Gate Purpose:
- **Validates Single-GPU Topology (`cuda:0` / `cuda:0`)** matching production A100.
- Verifies upstream Kaggle Dual-T4 report verdict is `PASS`.
- Exercises sequential memory release between Dense and Reranker.
- Emits `colab_t4_report.json` with verdict `PASS`.


In [ ]:
# ==============================================================================
# Cell 1: Hardware Preflight & Environment Loader
# ==============================================================================
import os
import sys
import torch
from pathlib import Path

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA GPU required for Colab T4 gate."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[+] Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
assert "T4" in gpu_name, f"Colab T4 gate requires Tesla T4 GPU (found {gpu_name})."

for env_path in [Path("/content/.env"), Path("/content/LegalIR/.env"), Path(".env")]:
    if env_path.is_file():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip("'\"")


In [ ]:
# ==============================================================================
# Cell 2: Repository Clone & Detached HEAD Checkout
# ==============================================================================
import subprocess
from pathlib import Path

EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA") or "3792b13699f4706c5a698b2d147a55e97fe4c0ce"
REPO_DIR = Path("/content/LegalIR") if Path("/content").exists() else Path.cwd()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR, check=False)
    print(f"[*] Checking out exact commit: {EXPECTED_COMMIT} (detached HEAD)...")
    subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[+] Working in: {REPO_DIR}")


In [ ]:
# ==============================================================================
# Cell 3: Dependencies & Dataset Setup
# ==============================================================================
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
sys.modules["torchao"] = None
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "pyvi", "pyarrow", "rank_bm25"], check=True)

dataset_dir = Path("/content/kaggle_dataset") if Path("/content").exists() else REPO_DIR / "artifacts/shared/canonical/v2"
if not (dataset_dir / "queries_train.parquet").is_file():
    dataset_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "phucdangg/legalir-task1-clean-data", "-p", str(dataset_dir), "--unzip"], check=True)
print(f"[+] Dataset verified at: {dataset_dir}")


In [ ]:
# ==============================================================================
# Cell 4: Execute Colab Single-T4 Gate (scripts/gates/run_colab_t4.py)
# ==============================================================================
from scripts.gates.run_colab_t4 import run_colab_t4_gate

output_dir = Path("/content/artifacts/task1/gates") if Path("/content").exists() else REPO_DIR / "artifacts/task1/gates"
k_report_path = REPO_DIR / "artifacts/task1/gates/kaggle_t4x2_report.json"

report = run_colab_t4_gate(
    dataset_dir=dataset_dir,
    output_dir=output_dir,
    expected_sha=EXPECTED_COMMIT,
    kaggle_report_path=k_report_path if k_report_path.is_file() else (output_dir / "kaggle_t4x2_report.json"),
    mock=False,
)
print(f"[+] Colab Single-T4 Gate execution verdict: {report.get('verdict')}")


In [ ]:
# ==============================================================================
# Cell 5: Assert Gate PASS
# ==============================================================================
import json

report_path = output_dir / "colab_t4_report.json"
assert report_path.is_file(), f"Report missing: {report_path}"
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report, indent=2))
assert report.get("verdict") == "PASS", f"Colab T4 Gate failed: {report}"
print("\n=================================================================")
print("[+] COLAB SINGLE-T4 GATE PASSED. READY FOR PRODUCTION A100 RUN.")
print("=================================================================")
